In [109]:
import os
import pandas as pd
import torch
import numpy as np
from os.path import join
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.models as models
from MaskRefiner import MaskRefinerUNet
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader, Dataset
import segmentation_models_pytorch as smp
from torch import nn

In [110]:
input_dir = "/Users/carricarte/PhD/Projects/MARL-med/scratch/dataset/data"
out_dir = "/Users/carricarte/PhD/Projects/MARL-med/scratch/output"
csv_path = "/Users/carricarte/PhD/Projects/MARL-med/scratch/dataset/data"
files = []
threshold = 0.815
[files.append(os.path.join(csv_path, f)) for f in os.listdir(csv_path) if f.endswith(".csv") and "._" not in f]
unique_images = []
try:
    for my_file in files:


        df = pd.read_csv(my_file)
        df_challenging_mask = df[df['IoU'] <= threshold]
        unique_images.append(pd.DataFrame(df_challenging_mask['image_file'].unique()))

    imgs_df = pd.concat(unique_images, ignore_index=True)
except:

        print("no unique images in file: ", my_file)

print(len(files))
print(len(imgs_df))

2
186


In [111]:
in_channels = 256
base_channels = 64
correction_scale = 0.3
dropout_rate = 0.1
use_attention = True
use_multi_scale = False
my_model = MaskRefinerUNet(in_channels, base_channels, correction_scale, use_attention, use_multi_scale, dropout_rate)

In [96]:
def get_embedding(img, embb_files):

    my_embedding = ""
    id = imgs_df.index[imgs_df[0] == img][0]
    file_no = id//100 + 1

    for f in embb_files:

        if f"{file_no:06d}" in f:

            my_file = np.load(f)

            try:

                my_embedding =  my_file[img]

            except:

                print(f"embedding not found in file: {file_no:06d}")

    return my_embedding


class MyDataset(Dataset):

    def __init__(self, input_dir):

        self._embedding_files = []
        self._mask_files = []

        [self._embedding_files.append(join(input_dir, emb_file)) for emb_file in os.listdir(input_dir) if "embedding" in emb_file and '._' not in emb_file and emb_file.endswith(".npz")]
        [self._mask_files.append(join(input_dir, mask_file)) for mask_file in os.listdir(input_dir) if "challenging_mask" in mask_file and '._' not in mask_file and mask_file.endswith(".npz")]

        self.images = np.load(self._mask_files[0])['image']
        self.mask_corrections = np.load(self._mask_files[0], allow_pickle=True)['mask']

        for mask_file in self._mask_files[1::]:

            data = np.load(mask_file)

            self.images = np.concatenate((self.images, data['image']))
            self.mask_corrections = np.concatenate((self.images, data['mask']))


    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        img = self.images[idx]
        mask_correction = self.mask_corrections[idx]
        embedding = get_embedding(img, self._embedding_files)

        # return embedding
        return embedding, mask_correction

def plot_training_history(train_losses, val_losses):
    """Plot training history"""
    fig, ax1 = plt.subplots(1, figsize=(6, 4))

    # Plot loss
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    plt.tight_layout()
    plt.savefig(join(out_dir, 'training_history_facial_emotions_improved.png'), dpi=150, bbox_inches='tight')
    print("Training history plot saved as 'training_history_facial_emotions_improved.png'")


def focal_loss(pred, target, alpha=0.75, gamma=2.0):
    """
    Focal Loss for binary classification/regression tasks.

    Args:
        pred: Predicted values (logits or probabilities), shape [B, H, W] or [B, ...]
        target: Ground truth values, same shape as pred
        alpha: Weighting factor for positive class (0-1)
        gamma: Focusing parameter (typically 1-5)

    Returns:
        Computed focal loss
    """
    # Convert to binary classification if continuous
    # Option 1: If pred/target are logits for binary classification
    bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
    p_t = torch.exp(-bce)  # Probability of correct class

    # Apply focal term
    focal_weight = (1 - p_t) ** gamma

    # Apply alpha balancing
    alpha_t = alpha * target + (1 - alpha) * (1 - target)

    loss = alpha_t * focal_weight * bce

    return loss.mean()


# def pure_delta_loss(pred_delta, true_delta):
#         """
#         Only supervise the delta itself - trust that correct deltas
#         lead to correct masks
#         """
#         focal_loss = -α * (1 - p_t)^γ * log(p_t)
#
#         # Focal loss for sparse delta prediction
#         focal_add = focal_loss(pred_delta[:, 0], true_delta[:, 0],
#                               alpha=0.8, gamma=2.5)
#         focal_remove = focal_loss(pred_delta[:, 1], true_delta[:, 1],
#                                  alpha=0.8, gamma=2.5)
#
#         # Optional: Penalize magnitude (encourages minimal changes)
#         sparsity = (torch.sigmoid(pred_delta).sum(dim=(1,2,3)) /
#                     pred_delta[0].numel()).mean()
#
#         return (focal_add + focal_remove) / 2 + 0.1 * sparsity

In [104]:
# annotations = get_annotation(mask_files)
my_dataset = MyDataset(input_dir)

train_size = int(0.7 * len(my_dataset))
val_size = int(0.15 * len(my_dataset))
test_size = int(0.15 * len(my_dataset))

if train_size + val_size + test_size != len(my_dataset):

  add_train = abs(train_size + val_size + test_size - len(my_dataset))
  train_size += add_train

my_train_dataset, my_val_dataset, my_test_dataset = random_split(my_dataset, [train_size, val_size, test_size])
pass

KeyboardInterrupt: 

In [ ]:
ratio = 10000
pos_weight = torch.tensor([ratio])  # e.g., if bg:fg = 100:1, use 100
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion = smp.losses.FocalLoss(mode='binary', alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(my_model.parameters(), lr=5e-4)
epochs = 15

In [ ]:
def train_epoch():

    my_model.train()
    losses = []

    for embedding, correction in tqdm(my_train_dataset):

        loss = criterion(my_model(embedding), correction)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        optimizer.zero_grad()

    return np.mean(losses)

def val_epoch():

    my_model.eval()
    losses = []

    with torch.no_grad():

        for embedding, correction in tqdm(my_val_dataset):

            losses.append(my_model(embedding), correction)

    return np.mean(losses)


In [ ]:
train_loss = []
val_loss = []

for epoch in range(epochs):

    train_loss.append(train_epoch())
    val_loss.append(val_epoch())

    # Print summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"Train Loss: {train_loss:.4f} | Validation Loss: {val_loss:.4f}")

plot_training_history(train_loss, val_loss)